<a href="https://colab.research.google.com/github/chaehyeonkim-lab/bioinformatics1/blob/main/Binformatics_term_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LIN28A가 결합하지만 번역 억제를 받지 않는 ER-associated mRNA 후보 탐색

논문에서의 흐름은 다음과 같다.

LIN28A가 mRNA에 결합한다. 특히 ER/membrane/secretory 관련 mRNA에 많이 결합하는데, 이들은 LIN28A을 knockdown한 경우 ribosome의 density가 증가하는 경향성을 보였다. 따라서 LIN28A는 ER-associated translation을 억제한다.

하지만 여기에는 분명히 예외가 존재한다.

LIN28A가 결합하는 ER-associated mRNA임에도 불구하고, Lin28a knockdown 후 ribosome density가 증가하지 않는 mRNA는 실제로 논문의 Figure 5.B에서도 확인 가능하다.


다음과 같이 그룹을 분류 가능하다.

LIN28A-bound ER mRNA
- CLIP enrichment가 높음
- localization annotation상 ER/membrane/secretory 관련

  1. Suppressed ER target
   - LIN28A-bound ER mRNA
   - Lin28a knockdown 후 ribosome density 증가
   - 원래 LIN28A가 번역을 억제했을 가능성이 높음

  2. Escapee ER target
   - LIN28A-bound ER mRNA
   - Lin28a knockdown 후 ribosome density가 증가하지 않음
   - LIN28A가 붙었지만 번역 억제 효과가 뚜렷하지 않음

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/hyeshik/colab-biolab.git
!cd colab-biolab && bash tools/setup.sh
exec(open('colab-biolab/tools/activate_conda.py').read())

Cloning into 'colab-biolab'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 76 (delta 26), reused 59 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 318.16 KiB | 1.74 MiB/s, done.
Resolving deltas: 100% (26/26), done.
./
./root/
./root/.profile
./root/.tmux.conf
./root/.vimrc
./root/.bin.priority/
./root/.bin.priority/pip2
./root/.bin.priority/pip
./root/.bin.priority/pip3
./root/.bashrc.biolab
./root/.condarc
--2026-05-21 08:36:09--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 163179296 (156M) [application/octet-stream]
Saving to: ‘miniconda3.sh’

miniconda3.sh       100%[===================>] 

In [3]:
!ls -al /content/drive/MyDrive/binfo1-datapack1/

total 7162852
-r-------- 1 root root 1370036258 Apr 27  2023 CLIP-35L33G.bam
-r-------- 1 root root    3118336 Apr 27  2023 CLIP-35L33G.bam.bai
-r-------- 1 root root       7113 May 11  2023 CLIP-let7g.bam
-r-------- 1 root root      14561 May 11  2023 CLIP-let7g-gene.pileup
-r-------- 1 root root    2685065 May 11  2023 CLIP-let7g.pileup
-r-------- 1 root root  883334756 Apr 27  2023 gencode.gtf
-r-------- 1 root root   24065406 Apr 27  2023 read-counts.txt
-r-------- 1 root root        751 Apr 27  2023 read-counts.txt.summary
-r-------- 1 root root 1003658801 Apr 27  2023 RNA-control.bam
-r-------- 1 root root    2276104 Apr 27  2023 RNA-control.bam.bai
-r-------- 1 root root 1260991122 Apr 27  2023 RNA-siLin28a.bam
-r-------- 1 root root    2710744 Apr 27  2023 RNA-siLin28a.bam.bai
-r-------- 1 root root  981684502 Apr 27  2023 RNA-siLuc.bam
-r-------- 1 root root    2606104 Apr 27  2023 RNA-siLuc.bam.bai
-r-------- 1 root root  737352902 Apr 27  2023 RPF-siLin28a.bam
-r-------- 1 r

In [7]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda install -y subread

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: \ | / done


==> WARNING: A newer version of conda exists. <==
    current version: 26.3.2
    latest version: 26.5.0

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

In [11]:
cnts = pd.read_csv('/content/drive/MyDrive/binfo1-datapack1/read-counts.txt', sep='\t', comment='#', index_col=0)

pseudo = 1
df = cnts.copy()

# read count가 너무 낮으면 비율 계산이 불안정하므로 제외
df = df[
    (df['RNA-control.bam'] > 10) &
    (df['RNA-siLuc.bam'] > 10) &
    (df['RNA-siLin28a.bam'] > 10)
].copy()

# LIN28A 결합 강도
df['clip_log2'] = np.log2(
    (df['CLIP-35L33G.bam'] + pseudo) /
    (df['RNA-control.bam'] + pseudo)
)

# control 조건의 ribosome density
df['rden_siLuc'] = (
    (df['RPF-siLuc.bam'] + pseudo) /
    (df['RNA-siLuc.bam'] + pseudo)
)

# Lin28a knockdown 조건의 ribosome density
df['rden_siLin28a'] = (
    (df['RPF-siLin28a.bam'] + pseudo) /
    (df['RNA-siLin28a.bam'] + pseudo)
)

# Lin28a knockdown 후 ribosome density 변화
df['rden_log2'] = np.log2(
    df['rden_siLin28a'] / df['rden_siLuc']
)

# RNA abundance 변화
df['rna_change_log2'] = np.log2(
    (df['RNA-siLin28a.bam'] + pseudo) /
    (df['RNA-siLuc.bam'] + pseudo)
)

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=['clip_log2', 'rden_log2', 'rna_change_log2'])